# Raw data EDA (before cleaning)
2026 Joey Manani & Anchorfish Team

Looks at the raw dataset file **before** cleaning

A big gap between features is **not** on its own a reason to drop something. A good feature is supposed to differ between type. 

Let's give them named (these are invented here, not actual terms):
1. **Same-site test:** is the feature present on the same site with and without the feature? Clean it out of the URL.
2. **Reality test:** is the rate believable for real traffic? Is it a reasonable value (human judgment)
3. **Research test:** is it consistent with actual research?

We will explain more these tests in the report, but same-site is quite important to understand, so I'll (Joey) provide a brief description:

Basically speaking, will the feature change depending on HOW the URL was actually stored? i.e from the set, is "http://www.phishingurl.com/" technically "worse" (phishing score) than "phishingurl.com"? These ARE the same site, so the feature should not be influenced by the way it's formatted in the URL.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import style  # shared colours and chart look, see style.py

style.use()

DATA = Path("malicious_phish.csv") # provided Kaggle set
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

SCHEME = r"^[a-zA-Z][a-zA-Z0-9+.-]*://"  # http://, https://

df = pd.read_csv(DATA)
# same names as the cleaned data (benign -> Legitimate), in a fixed order so every chart and table lines up
df["type"] = pd.Categorical(df.type.map(style.KAGGLE_NAMES), categories=style.TYPE_ORDER)
df.type.value_counts()

## 0. Quick health check
The Week 6 lecture's "five commands": look at a few rows, the column types, missing values, exact duplicates and a summary, first.

In [ ]:
print(df.head(3), "\n")
print(df.dtypes, "\n")
print("missing values:", df.isnull().sum().sum())
print("exact duplicate rows:", df.duplicated().sum())  # broken down by type in section 4
df.describe()  # for text columns: how many, how many different, the most common value

## 1. URL formatting by type
Shown as **% of each type**, not numbers: there are about 5x more legit URLs so we gotta scale them accordingly.

In [ ]:
bare = df.url.str.replace(SCHEME, "", regex=True)  # so "http://www.x.com" still counts as starting with www


flags = pd.DataFrame({
    "type": df.type,
    "has scheme": df.url.str.match(SCHEME),
    "starts with www.": bare.str.match(r"(?i)www\d*\."),
    "is IP host": bare.str.match(r"(\d{1,3}\.){3}\d{1,3}"), # ipv4
})

# based on the flags, calculate percentages of the totals that have these features, and group these by their URL type
percentage_table = flags.groupby("type").mean() * 100

# must be transposed for plotting (one group per flag, one bar per type)
ax = percentage_table.T.plot.bar(rot=0, width=0.7, color=style.colours(percentage_table.index),
                                 ylabel="% of URLs in type", title="Formatting by type (raw)")
ax.legend(title=None)
ax.grid(False, axis="x")  # pandas turns on both grids, keep only the value axis
style.save(ax.figure, FIGURES / "raw_flags_by_type.png")
plt.show()
percentage_table.round(2)

So according to the output above, only 8% of the benigns have a scheme, compared to 26% of the phishing URLs. A model would treat the scheme as a strong indicator of malicious, but to me, a human, this is unreliable: every real URL has a scheme, so 8% says more about how the benign list was stored than about the sites themselves (reality test). The scheme gets stripped during cleaning.

The same pattern is observed for the other feature including whether it starts with www. 39% of phishing URLs start with www. compared to only 3% of the benigns, meaning a trained model will pick up that anything with www probably means phishing. This fails the same-site test (`www.x.com` is the same site as `x.com`) and the reality test, so instead of down-weighting it we clean www. out of every URL before extracting features. Just dropping the `has_www` column isn't enough, because www. also adds a dot and dot_count would pick it up.

IP host is interesting. 0.01% of the benigns are an IPv4 host compared to 0.35% of the phishing. This makes sense as a human since most real actual websites have a proper domain instead of just an IP. Malware is even better, where 36% of them are IP-based hosts which makes sense since purchasing a domain 1. costs money, 2. has an ownership record to make it traceable, 3. has to wait for DNS propagation to even work (>1 day)

### 1b. A few more formatting flags
Same idea as above with four more: is it https, does it end with a `/`, does it have a `?query`, and does it have any uppercase letters.

In [ ]:
more_flags = pd.DataFrame({
    "type": df.type,
    "https": df.url.str.match(r"(?i)https://"),
    "ends with /": df.url.str.endswith("/"),
    "has ?query": df.url.str.contains("?", regex=False),
    "has uppercase": df.url.str.contains("[A-Z]"),
})

more_percentage_table = more_flags.groupby("type").mean() * 100

ax = more_percentage_table.T.plot.bar(rot=0, width=0.7, color=style.colours(more_percentage_table.index),
                                      ylabel="% of URLs in type", title="More formatting by type (raw)")
ax.legend(title=None)
ax.grid(False, axis="x")
style.save(ax.figure, FIGURES / "raw_flags_more_by_type.png")
plt.show()
more_percentage_table.round(2)

Defacement is the odd one out here. From the first chart 100% of them have a scheme, but 0% are https and 0% end with a `/`. It appears that defacement URLs all were sourced from one place that stored them one specific way. This fails the same-site test: a model would learn how that source formats its URLs not anything about defacement. (Irrelevant to phishing but still important to understand and see here)

https is backwards too. Only 0.46% of the benigns are https, compared to 20.8% of malware and 7.4% of phishing. About 92% of the benigns have no scheme at all, so they cannot be https in the first place. A model trained on this would learn https = bad, which fails the reality test. So https has the same problem as scheme and www. (https *is* a scheme, makes sense anyways)

Query and uppercase differ a lot as well (57% of defacement has a query, 55% of malware has uppercase vs 18% of phishing). These are more about the page than about how the URL got written down, so a gap here is interesting but not necessarily a problem. Often times, these URLs have URL encodings which *are* uppercase by design, so while its not immediately bad data, its still a convention that cannot reasonably differentiate between the different URL types since both legit and malicious sites use uppercase characters for encoding. The sake is, *does it have encoding* in the first place?

## 2. URL length, legit vs phishing

In [ ]:
length = df.url.str.len()
LENGTH = range(0, 200, 1)

fig, ax = plt.subplots()
for url_type in ["Legitimate", "Phishing"]: # do this for the legits and phishing urls; create a histogram with outline
    colour = style.TYPE_COLOURS[url_type]
    lengths = length[df.type == url_type]

    # density, not number of again
    ax.hist(lengths, bins=LENGTH, density=True, histtype="stepfilled", color=colour, alpha=0.12, edgecolor="none")  # light wash
    ax.hist(lengths, bins=LENGTH, density=True, histtype="step", color=colour, linewidth=0.3, label=url_type)  # outline

# label the two spikes explained further down (section 2 notes and section 5)
for url_type, spike, text in [("Legitimate", 31, "31: YouTube videos"), ("Phishing", 27, "27: ietf.org RFC pages")]:
    height = (length[df.type == url_type] == spike).mean()
    ax.annotate(text, xy=(spike + 0.5, height), xytext=(spike + 25, height), color=style.INK_SOFT, fontsize=9,
                va="center", arrowprops={"arrowstyle": "-", "color": style.MUTED, "linewidth": 0.8})

ax.set_xlabel("URL length (characters, limit @ 200)")
ax.set_ylabel("Share of URLs per character")
ax.set_title("URL length by type (raw)")
ax.legend()
style.save(fig, FIGURES / "raw_length_by_type.png")
plt.show()
length.groupby(df.type).median()

### Certain things to play with

- Can set the upper value to 500, but above 200 is mostly outliers. 200 is fine, step = 1. This shows a pretty wide distribution of the URL lengths and is cool to see most is within a certain range.
- If set density to False, look at the massive spike at 31 (benign), over half of it (8,545 of 15,737) is legitimate YouTube URLs. Nearly all of those are video links: `youtube.com/watch?v=` plus the 11 character video ID is exactly 31 characters. Cool
- The same large spike at phishing url's 27 length looked like genuine random phishing bunching at 27... Wow. Turns out it isn't: section 5 shows it's one mislabelled site.

### Interestingly...

So according to this set of data, actually, the PHISHING URLs are shorter than the benign ones. This seems counterintuitive at first glance, but it could be due to the way the dataset was created. This is really interesting to know, and will therefore mean that URL length could be a useful feature for distinguishing between benign and phishing URLs - but we need to be careful not to overfit to this single feature since clearly the counterintuitiveness means it could be either way. This histogram revealed a lot actually. (I, Joey, also played a significant amount using Excel's inbuilt sort and filter functions - we need to do this programmatically as well, therefore this notebook exists)

### 2b. Length outliers, all four types
The histogram above stops at 200 and only has benign and phishing, so the very long and very short URLs are hidden. A box plot on a log scale shows all four types. The whiskers are set to 1% and 99%, so every dot is in the most extreme 2% of its type.

In [ ]:
fig, ax = plt.subplots()
boxes = ax.boxplot(
    [length[df.type == t] for t in style.TYPE_ORDER],
    tick_labels=style.TYPE_ORDER,
    whis=(1, 99),  # whisker cap at 1% and 99% (lower than 1% of data; higher than 99% of data are outlier)
    widths=0.5,
    patch_artist=True,
    whiskerprops={"color": style.INK_SOFT},
    capprops={"color": style.INK_SOFT},
    flierprops={"markersize": 2, "alpha": 0.3, "markeredgecolor": "none"},
)
for box, outliers, t in zip(boxes["boxes"], boxes["fliers"], style.TYPE_ORDER):
    box.set(facecolor=style.TYPE_COLOURS[t] + "33", edgecolor=style.TYPE_COLOURS[t], linewidth=1.5)
    outliers.set(markerfacecolor=style.TYPE_COLOURS[t])

ax.set_yscale("log")
ax.set_ylabel("URL length (characters, log scale)")
ax.set_title("URL length by type, whiskers at 1% and 99% (raw)")
style.save(fig, FIGURES / "raw_length_outliers_by_type.png")
plt.show()
length.groupby(df.type).describe(percentiles=[0.01, 0.5, 0.99]).round(0)

In [ ]:
df[length <= 5]  # the shortest "URLs" in the whole set

Seems theres some malformed URLs in the dataset too... We will clean them later!


The boxes (the middle 50%) sit in a similar place for all four, so a normal URL is not wildly different between types. The tails are where it gets weird:

- Benign goes up to 2,175 characters and phishing to 1,779, while defacement and malware stop at 327 and 378.
- Phishing goes down to 1 character. The table above is every URL of 5 characters or less: all 8 are labelled phishing and most of them are broken bytes, not URLs. These need to be cleaned out.

The outliers are not many rows (99% of every type is under 240 characters).

## 3. Top TLDs by type
In `eda_features.ipynb`, because `tld` column only exists in the extracted features.

## 4. Duplicates and conflicting labels
How many rows are an exact copy of an earlier row (same URL and same type), as **% of each type**.

In [ ]:
is_duplicate = df.duplicated()  # same url AND same type as an earlier row

duplicate_table = pd.DataFrame({
    "duplicate rows": is_duplicate.groupby(df.type).sum(),
    "% of type": is_duplicate.groupby(df.type).mean() * 100,
})

ax = duplicate_table["% of type"].plot.bar(rot=0, xlabel="", width=0.6, color=style.colours(duplicate_table.index),
                                           ylabel="% of URLs in type", title="Exact duplicate rows by type (raw)")

ax.grid(False, axis="x")
ax.bar_label(ax.containers[0], fmt="%.2f%%", padding=3, color=style.INK_SOFT, fontsize=9)
style.save(ax.figure, FIGURES / "raw_duplicates_by_type.png")
plt.show()
duplicate_table.round(2)

In [ ]:
# same URL, but labelled as two different types (12) not many to filter
labels_per_url = df.groupby("url").type.nunique()
df[df.url.isin(labels_per_url[labels_per_url > 1].index)].sort_values("url")

10,066 rows are exact copies, and 8,875 of those are malware. That is 27% of the malware, compared to 0.01% of the benigns. So malware is really about 23,600 different URLs and not 32,520, and it was already the smallest type.

If the copies stay in, the same URL can land in both the training set and the test set, and the score will look better than it really is. They need to go before any split.

The second table is the 6 URLs that are in the set twice with two different labels, benign and phishing every time. It's only 12 rows, but a Wikipedia article cannot be both. More on this in section 6.

## 5. How many URLs is each host?
650k URLs is not 650k websites. Here the host is pulled out of each URL (everything before the first `/`, lowercased, without the `www.`) to see how many different sites each type is made of.

Hypothesis: some hosts will be packed with a bunch of URLs from the same type

In [ ]:
host = (bare.str.split(r"[/?#]", n=1, regex=True).str[0].str.lower().str.replace(r"^www\d*\.", "", regex=True))

by_type = host.groupby(df.type)
concentration = pd.DataFrame({
    "URLs per host": by_type.size() / by_type.nunique(),
    "top 10 hosts (% of type)": by_type.apply(lambda h: h.value_counts(normalize=True).head(10).sum() * 100),
})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, column in zip(axes, concentration.columns):
    concentration[column].plot.bar(ax=ax, rot=0, xlabel="", width=0.6, color=style.colours(concentration.index))
    ax.set_title(column, fontsize=11)
    ax.grid(False, axis="x")
    ax.bar_label(ax.containers[0], fmt="%.1f", padding=3, color=style.INK_SOFT, fontsize=9)

fig.tight_layout()
style.save(fig, FIGURES / "raw_host_concentration.png")
plt.show()

In [ ]:
# Table: rank x type, each cell "host (pct of type%)"
pd.DataFrame(
    {t: [f"{h} ({v:.2f}%)" for h, v in host[df.type == t].value_counts(normalize=True).mul(100).head(5).items()] for t in style.TYPE_ORDER},
    index=range(1, 6),
).rename_axis("rank")

Defacement is 96,457 URLs from only 2,123 hosts, so about 45 URLs per site. Phishing is the opposite at 1.7. With a random train/test split, pages from the same defaced site end up on both sides and the model only needs to remember the site. The host COULD be split into different sets...

Malware's top 10 hosts are 34% of it, and just two of them (`9779.info` and `mitsui-jyuku.mixh.jp`) are 21%. Whatever those two sites look like is what the model will think malware looks like. Overfitting....

The phishing column fails the reality test. The biggest "phishing" host is `tools.ietf.org` at 3.4%, which is where the internet standards (RFCs) are published, and Wikipedia is 4th. There's a bit from `sourceforge.net` too, but not as much. 

This also explains the spike at length 27 in section 2. Example: `tools.ietf.org/html/rfc1951` is 27 characters, and 2,333 of the 4,584 phishing URLs with length 27 are from IETF. Without them, 27 is the same height as its neighbours (smoother bell curve). So that spike is not phishing bunching up, it is one mislabelled site from the data set.

## 6. Hosts with more than one label
Following on from the 6 conflicting URLs: a whole host can be under more than one type too. These are the 15 hosts with the most URLs that disagree with the host's main label.

In [ ]:
label_mix = pd.crosstab(host, df.type)
mixed = label_mix[(label_mix > 0).sum(axis=1) > 1]  # hosts that are under more than one type
disagree = mixed.sum(axis=1) - mixed.max(axis=1)  # URLs that are not the host's main label
worst = mixed.loc[disagree.nlargest(15).index]

# each host as 100%, [::-1] so the worst one is at the top
ax = worst.div(worst.sum(axis=1), axis=0).mul(100).iloc[::-1].plot.barh(
    stacked=True, figsize=(10, 5), width=0.7, color=style.colours(worst.columns), xlabel="% of the host's URLs", ylabel="",
    title=f"Label mix of the 15 most mixed hosts ({len(mixed)} hosts have more than one label)",
)

ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
style.save(ax.figure, FIGURES / "raw_mixed_label_hosts.png")
plt.show()
worst

4,094 hosts are under more than one type, and the top 15 are pretty much all benign vs phishing.

`pastehtml.com`, `docs.google.com`, `sites.google.com`, `angelfire.com` and `bit.ly` let anyone put up anything, so these hosts have both good and bad pages. Therefore, the host tells the model nothing and it has to come from the rest of the URL.

Some of it is not believable. `en.wikipedia.org` has 538 phishing URLs and `sourceforge.net` is more phishing (536) than benign (441). Together with `tools.ietf.org` in section 5, it looks like part of the phishing set is mislabelled benign pages. We have a LOT of cleaning to do!